# Brainstorming and Focus Group Quantitative Experimentation 2.1: :**Difficult people** under **divergence intervention** only

Can we use TinyTroupe to brainstorm product ideas?

In [1]:
import sys

from pprint import pprint

from tinytroupe.agent import TinyPerson
from tinytroupe.environment import TinyWorld
from tinytroupe.experimentation import InPlaceExperimentRunner
from tinytroupe.steering import Intervention
from tinytroupe.examples import *
from tinytroupe.validation import propositions
from tinytroupe.extraction import ResultsExtractor
from tinytroupe.utils.parallel import parallel_map_dict, parallel_map_cross
from tinytroupe.validation import hard_persona_adherence, persona_adherence, self_consistency, fluency, task_completion, divergence

# specific utilities
from common_utils import *


!!!!
DISCLAIMER: TinyTroupe relies on Artificial Intelligence (AI) models to generate content. 
The AI models are not perfect and may produce inappropriate or inaccurate results. 
For any serious or consequential use, please review the generated content before using it.
!!!!

Looking for default config on: C:\Users\pdasilva\repos\TinyTroupe and personal repos\TinyTroupe\tinytroupe\utils\..\config.ini
Found custom config on: c:\Users\pdasilva\repos\TinyTroupe and personal repos\TinyTroupe\publications\paper_artifacts_april-2026\config.ini
TinyTroupe version: 0.8.0
Current date and time (local): 2026-04-29 08:11:07
Current date and time (UTC):   2026-04-29 11:11:07

Current TinyTroupe configuration 
[OpenAI]
api_type = azure
azure_api_version = 2024-12-01-preview
model = gpt-5-mini
reasoning_model = o3-mini
vision_detail = auto
embedding_model = text-embedding-3-small
azure_embedding_model_api_version = 2023-05-15
max_completion_tokens = 128000
timeout = 300
max_attempts = 5
waiting_tim

## Parameters

In [2]:
full_mode = True  # set to True to run the full mode with all agents and tasks

# avoid displaying the communication, to make the output cleaner for eval
TinyPerson.communication_display = False

In [3]:
if full_mode:
    repetitions_per_task = 2
    simulation_steps = 5
    qty_agents = 12
    qty_proposals = 4

else:
    repetitions_per_task = 2
    simulation_steps = 5
    qty_agents = 4
    qty_proposals = 1


## Experiment setup

In [4]:
experiment_runner = InPlaceExperimentRunner("./brainstorming_and_focus_group_quantitative_experimentation_2.2.json")

experiment_runner.add_experiment("Control")
experiment_runner.add_experiment("Treatment")

In [5]:
experiment_runner.activate_next_experiment()

#experiment_runner.fix_active_experiment("Control")
#experiment_runner.fix_active_experiment("Treatment")

In [6]:
print(f"Running experiment {experiment_runner.get_active_experiment()}")

Running experiment Control


## Agents and populations

In [7]:

people = []
if not experiment_runner.has_finished_all_experiments():
    # load agents
    people = TinyPerson.load_specifications_from_folder("./population/difficult_people_2")

    # filter to make it go faster?
    if qty_agents is not None:
        people = people[:qty_agents]

    # customize and print minibios 
    for person in people:

        person.import_fragment("./fragments/difficult_person.agent.fragment.json")

        # disable quality checks for both Control and Treatment
        person.action_generator.enable_quality_checks = False

        print(person.minibio(extended=False))


Alan Merrick is a 48 year old Administrative Officer (Benefits and Records), British, currently living in Manchester, United Kingdom.
Anthony Russo is a 42 year old Journeyman Electrician / Senior Field Technician, American, currently living in Cleveland, Ohio, USA.
Anya Calder-Mori is a 45 year old Freelance Graphic Designer, Conceptual Artist and Cultural Critic, British, currently living in Camberwell, London, UK.
Barbara Jean Pratt is a 68 year old Retiree (former assembly line worker / part-time volunteer at church thrift shop), American, currently living in Small town near Toledo, Ohio, USA.
Colin Arthur Matthews is a 42 year old Operations Manager (Mid-level), British, currently living in Manchester, UK.
Colin Murray is a 52 year old Benefits and Housing Support Officer, British, currently living in Salford, Greater Manchester, UK.
Connor Walsh is a 28 year old Senior Customer Service Associate / Shift Lead (Retail Grocery Chain), American, currently living in Cleveland, Ohio, U

In [8]:
len(people)

12

In [9]:
# divide people in several groups of 5
people_groups = []
for i in range(0, len(people), 4):
    people_groups.append(people[i:i+4]
    )

len(people_groups)

3

In [10]:
# In this experiment, we'll not use action correction. We'll instead experiment only with divergence intervention.
for person in people:
    person.action_generator.enable_reasoning_step = False
    person.action_generator.enable_quality_checks = False

## Proposals

In [11]:
proposals = [
    {"theme": "Daily Life and Convenience",
     "objective": "Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions."},

    {"theme": "Personal Growth and Wellbeing",
     "objective": "Generate concepts for products or experiences that support personal development, health, mental wellness, emotional care, or community connection."},

    {"theme": "Discovery and Exploration",
     "objective": "Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self."},

    {"theme": "Productivity and Resourcefulness",
     "objective": "Invent new tools, processes, or organizational systems that empower people or groups to achieve more, optimize resources, or collaborate effectively."},

    {"theme": "Creativity and Expression",
     "objective": "Design ideas for new products, platforms, or services that inspire creativity, foster expression, enhance artistic skills, or enable new forms of storytelling and communication."}
]

if not full_mode:
    proposals = proposals[:qty_proposals]

In [12]:
# divide the proposals in exactly two groups (half/half)
proposals_groups = []
proposals_groups.append(proposals[:len(proposals)//2])
proposals_groups.append(proposals[len(proposals)//2:])

proposals_groups

[[{'theme': 'Daily Life and Convenience',
   'objective': 'Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.'},
  {'theme': 'Personal Growth and Wellbeing',
   'objective': 'Generate concepts for products or experiences that support personal development, health, mental wellness, emotional care, or community connection.'}],
 [{'theme': 'Discovery and Exploration',
   'objective': 'Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.'},
  {'theme': 'Productivity and Resourcefulness',
   'objective': 'Invent new tools, processes, or organizational systems that empower people or groups to achieve more, optimize resources, or collaborate effectively.'},
  {'theme': 'Creativity and Expression',
   'objective': 'Design ideas for new products, platforms, or services that inspire creativity, foster expression, enhance art

## Auxiliary functions

In [13]:
def brainstorming_battery(agents, proposals, interventions, agent_propositions, environment_propositions, 
                          repetitions = 5, simulation_steps=10): 
    
    agent_propositions_scores = {}
    environment_propositions_scores = {}

    experiments_count = 0
    total_expected_experiments = len(proposals) * repetitions #* len(agents)

    # loop over proposals and repetitions
    for proposal in proposals:

        objective = proposal["objective"]
        theme = proposal["theme"]

        for i in range(repetitions):
            print("\n############## STARTING A NEW RESEARCH SESSION #################")
            print(f"Overall experiment number: {experiments_count+1} / {total_expected_experiments}")
            print(f"Discussion objective: {objective}")
            print(f"Trial number: {i+1}")
            print(f"Agents: {agents}")

            # clear the episodic memory of all agents
            for person in agents:
                person.clear_episodic_memory()

            world = TinyWorld(agents=agents, interventions=interventions)
            
            # Participants introduce themselves
            world.broadcast(f"""
                Hello everyone! Let's start by introducing ourselves, and mentioning problems we face in our daily personal
                and professional lives related to the following theme: {theme}
                
                Please:
                  - present yourself and your background;
                  - present some key personal problems related to the theme;
                  - present some key problems related to the theme that you face in your work;
                  - present some key problems related to the theme that you see in your industry as a whole.
                  
                Don't discuss solutions yet, just the problems you face and see others facing.
                """)
            world.run(1)
            
            # now to the brainstorming session itself
            world.broadcast(f"""
                Folks, your mission is to brainstorm {objective}. 
                Please follow these guidelines:
                  - give a unique and informative name to each idea you propose, so that it is easy to refer to it. Say it like "Idea name: '<name of the idea>'".;
                  - explain why you think it is a good idea, and what problem it solves, and how you feel about it;
                  - your ideas should be new complete, self-contained, products or services, not features for other existing products or services;
                  - think of creative ideas that would somehow help you in both in your personal and professional lives.
                  - create as many different and unique ideas as you can during the brainstorming session. Each idea must be **completely** different from the others 
                    (either by yourself or by others), and not just a variation of an existing idea.                    
                  - you should criticize each other's ideas, in order to make sure they are as
                    good as possible, but no more than once per idea.
                  - you should also provide suggestions for improvement to each other's ideas, in order to make them as good as possible, 
                    but no more than once per idea.
                  - regardless of critique or complement, you **must** primarily propose new ideas quickly, 
                    not just build on existing ones. 
                  - propose one idea at a time, instead of proposing multiple ideas at once, to allow appropriate discussion.
                  - you should **not** propose ideas that are too similar to each other, or to the ones already proposed by others.
                  - before saying anything, THINK deeply about yourself, your beliefs, interests, needs, life, etc., to come up with ideas that are
                    truly unique and different from the ones already proposed by others.
                   
                Please start the discussion now.
                """)
            world.run(simulation_steps)

            # extract and count ideas
            rapporteur = agents[0]  # the first agent is the rapporteur
            rapporteur.listen_and_act("Can you please consolidate the ideas that the group came up with? Provide a lot of details on each idea, and complement anything missing.")
            ideas = ResultsExtractor().extract_results_from_agent(rapporteur, 
                                    extraction_objective="Consolidates the ideas that the group came up with, explaining each idea as an item of a list." \
                                                        "Add information about: what problem the idea solves; to which target audience it is meant." \
                                                        "how is it different from competing, existing, products.", 
                                    situation="A focus group to brainstorm new product ideas.",
                                    fields= ["name", "description", "problem", "target_audience", "competition_analysis"],
                                    fields_hints={"ideas": "must be the root of the resulting dictionary."},)
            pprint(ideas)
            if "ideas_qty" not in environment_propositions_scores:
                environment_propositions_scores["ideas_qty"] = []
            if ideas is not None and "ideas" in ideas and isinstance(ideas["ideas"], list):
                environment_propositions_scores["ideas_qty"].append(len(ideas["ideas"]))

            # Evaluate environment propositions in parallel
            env_results = parallel_map_dict(
                environment_propositions,
                lambda item: item[1].copy().score(
                    world, 
                    claim_variables={"task_description": f"A brainstorming or focus group session was run about: {objective}."}, 
                    return_full_response=True
                )
            )
            
            # Process environment results
            for k, result in env_results.items():
                if k not in environment_propositions_scores:
                    environment_propositions_scores[k] = []
                environment_propositions_scores[k].append(result["value"])
                print("value: ", result["value"])
                print("justification: ", result["justification"])
                print("reasoning: ", result["reasoning"])

            # Evaluate agent propositions across all agents in parallel
            agent_results = parallel_map_cross(
                [agents, agent_propositions.items()],
                lambda agent, prop_item: (
                    prop_item[0],  # proposition key
                    prop_item[1].copy().score(agent, return_full_response=True)  # result
                )
            )
            
            # Process agent results
            for k, result in agent_results:
                if k not in agent_propositions_scores:
                    agent_propositions_scores[k] = []
                if result is not None:
                    agent_propositions_scores[k].append(result["value"])
                    print("value: ", result["value"])
                    print("justification: ", result["justification"])
                    print("reasoning: ", result["reasoning"])
                    print("\n\n")
                else:
                    print(f"*****WARNING:***** Agent did not respond to proposition {k}.")
            #
            ##for k, proposition in agent_propositions.items():
            ##    for person in world.agents:
            ##        result = proposition.copy().score(person, return_full_response=True)
            ##        
            ##        if k not in agent_propositions_scores:
            ##            agent_propositions_scores[k] = []
            ##        agent_propositions_scores[k].append(result["value"])
            ##
            ##        print("value: ", result["value"])
            ##        print("justification: ", result["justification"])
            ##        print("reasoning: ", result["reasoning"])
            ##        print("\n\n")
            ##
            
            experiments_count += 1
            print("\n\n")

    return agent_propositions_scores, environment_propositions_scores



## Perform experiment

In [14]:
agent_propositions_scores={}
environment_propositions_scores={}

In [15]:
def brainstorm(people, proposals=proposals):
    global agent_propositions_scores, environment_propositions_scores
    if not experiment_runner.has_finished_all_experiments():

        interventions = []
        if experiment_runner.get_active_experiment() == "Treatment":
            interventions = \
                Intervention.create_for_each(people)\
                    .set_functional_precondition(lambda target: target.actions_count >=7)\
                    .set_textual_precondition(
                        """
                        AGENT IS NOT PROPOSING COMPLETELY NEW PRODUCT/SERVICE IDEAS ANYMORE:
                        The last **entirely** new product/service idea proposed by this agent, if any, was proposed by him/her **more** than 5 of simulation events ago.
                        That is to say, the agent has not proposed any new product/service idea in the last 5 of his/her simulation trajectory events.
                        Additional features, variations of or other refinements to product/service ideas already proposed are NOT considered new!

                        How to compute the steps gap:
                        1. Determine the current next event number (N); and the last event number in which the agent proposed a new product/service idea (M).
                            This information can be found in the simulation trajectory.
                        2. Compute the **difference** beteween the current next event number and the last event number in which the agent proposed a new product/service idea: D = N - M
                        3. The proposition is true if, and only if, the difference D is **greater than** 5.
                        """)\
                    .set_effect(lambda target: target.think("""
                                                            I need to propose additional, **completelly** new and different, product/service ideas. This was part of the requirement for this session.
                                                            I will propose an entirely **new** idea now, I **cannot** repeat or refine previous ideas! I cannot make variations
                                                            of previous ideas (e.g., "XYZ for A", "XYZ for B", "XYZ for Z" are repetitive, there should be only one "XYZ"), 
                                                            I need to think of something **entirely** new and different.
                                                            To help me avoid repeating previous ideas, I'll now explicitly THINK about all the ideas already given by myself or
                                                            others, and then, based on that, I'll think again about a new unique idea.
                                                            """))

                                                            
        tmp_agent_propositions_scores, tmp_environment_propositions_scores = \
            brainstorming_battery(
                agents=people,
                proposals=proposals,
                interventions=interventions,    
                agent_propositions={
                    "Hard Persona Adherence": hard_persona_adherence,
                    "Self-consistency": self_consistency,
                    "Fluency": fluency
                },
                environment_propositions={
                    "Task Completion": task_completion,
                    "Divergence": divergence
                },
                repetitions=repetitions_per_task,
                simulation_steps=simulation_steps
            )

        pprint("NEW AGENT PROPOSITIONS SCORES")
        pprint(tmp_agent_propositions_scores)
        print("\n\n")
        pprint("NEW ENVIRONMENT PROPOSITIONS SCORES")
        pprint(tmp_environment_propositions_scores)

        # merge the scores lists
        agent_propositions_scores = merge_dicts_of_lists(tmp_agent_propositions_scores, agent_propositions_scores)
        environment_propositions_scores = merge_dicts_of_lists(tmp_environment_propositions_scores, environment_propositions_scores)

        return agent_propositions_scores, environment_propositions_scores

In [16]:
brainstorm(people_groups[0], proposals_groups[0]) if len(people_groups) > 0  and len(proposals_groups) > 0 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 4
Discussion objective: Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.
Trial number: 1
Agents: [TinyPerson(name='Alan Merrick'), TinyPerson(name='Anthony Russo'), TinyPerson(name='Anya Calder-Mori'), TinyPerson(name='Barbara Jean Pratt')]
2026-04-29 08:12:24,310 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 1] Running world simulation step 1 of 1.


───────────────────────────────────────────── TinyWorld 1 step 1 of 1 ─────────────────────────────────────────────

2026-04-29 08:12:24,326 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 1] No timedelta provided, so the datetime was not advanced.
2026-04-29 08:12:27,814 - ThreadPoolExecutor-0_3(40468) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 08:12:27,821 - ThreadPoolExecutor-0_1(15484) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 08:12:27,836 - ThreadPoolExecutor-0_0(5840) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 08:12:27,838 - ThreadPoolExecutor-0_2(31180) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 08:12:28,433 - ThreadPoolExecutor-0_1(15484) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 08:12:28,438 - ThreadPoolExecutor-0_3(40468) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 08:12:28,441 - ThreadPoolExecutor-0_2(31180) - tinytroupe - INFO - Waiting 

───────────────────────────────────────────── TinyWorld 1 step 1 of 5 ─────────────────────────────────────────────

2026-04-29 08:13:15,835 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 1] No timedelta provided, so the datetime was not advanced.
2026-04-29 08:13:17,793 - ThreadPoolExecutor-1_1(38344) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 08:13:17,806 - ThreadPoolExecutor-1_3(29268) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 08:13:17,839 - ThreadPoolExecutor-1_1(38344) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 08:13:17,846 - ThreadPoolExecutor-1_3(29268) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 08:13:17,855 - ThreadPoolExecutor-1_0(35400) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 08:13:17,861 - ThreadPoolExecutor-1_2(15488) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 08:13:17,911 - ThreadPoolExecutor-1_0(35400) - tinytroupe - INFO - Waiting

...

───────────────────────────────────────────── TinyWorld 4 step 4 of 5 ─────────────────────────────────────────────

2026-04-29 08:57:58,923 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 4] No timedelta provided, so the datetime was not advanced.
2026-04-29 08:58:00,571 - ThreadPoolExecutor-28_1(12780) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 08:58:00,590 - ThreadPoolExecutor-28_3(31580) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 08:58:00,620 - ThreadPoolExecutor-28_2(24776) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 08:58:00,627 - ThreadPoolExecutor-28_0(36740) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 08:58:00,664 - ThreadPoolExecutor-28_1(12780) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 08:58:00,670 - ThreadPoolExecutor-28_3(31580) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 08:58:00,698 - ThreadPoolExecutor-28_2(24776) - tinytroupe - INFO - 

───────────────────────────────────────────── TinyWorld 4 step 5 of 5 ─────────────────────────────────────────────

2026-04-29 08:58:57,174 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 4] No timedelta provided, so the datetime was not advanced.
2026-04-29 08:58:58,709 - ThreadPoolExecutor-29_0(39992) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 08:58:58,768 - ThreadPoolExecutor-29_0(39992) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 08:58:58,804 - ThreadPoolExecutor-29_3(20636) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 08:58:58,817 - ThreadPoolExecutor-29_2(8220) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 08:58:58,823 - ThreadPoolExecutor-29_1(31176) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 08:58:58,881 - ThreadPoolExecutor-29_3(20636) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 08:58:58,884 - ThreadPoolExecutor-29_2(8220) - tinytroupe - INFO - Wa

({'Hard Persona Adherence': [3, 3, 0, 4, 0, 1, 1, 5, 1, 3, 0, 5, 3, 0, 1, 4],
  'Self-consistency': [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 7],
  'Fluency': [7, 7, 8, 1, 7, 7, 8, 8, 8, 8, 8, 8, 7, 9, 9, 7]},
 {'ideas_qty': [4, 4, 3, 4],
  'Task Completion': [9, 9, 9, 9],
  'Divergence': [1, 0, 0, 0]})

In [17]:
brainstorm(people_groups[0], proposals_groups[1]) if len(people_groups) > 0  and len(proposals_groups) > 1 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 6
Discussion objective: Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.
Trial number: 1
Agents: [TinyPerson(name='Alan Merrick'), TinyPerson(name='Anthony Russo'), TinyPerson(name='Anya Calder-Mori'), TinyPerson(name='Barbara Jean Pratt')]
2026-04-29 09:14:06,257 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 5] Running world simulation step 1 of 1.


───────────────────────────────────────────── TinyWorld 5 step 1 of 1 ─────────────────────────────────────────────

2026-04-29 09:14:06,265 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 5] No timedelta provided, so the datetime was not advanced.
2026-04-29 09:14:08,083 - ThreadPoolExecutor-32_3(23124) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 09:14:08,115 - ThreadPoolExecutor-32_2(28764) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 09:14:08,123 - ThreadPoolExecutor-32_0(40196) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 09:14:08,134 - ThreadPoolExecutor-32_1(41156) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 09:14:08,148 - ThreadPoolExecutor-32_3(23124) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 09:14:08,181 - ThreadPoolExecutor-32_2(28764) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 09:14:08,198 - ThreadPoolExecutor-32_0(40196) - tinytroupe - INFO - 

───────────────────────────────────────────── TinyWorld 5 step 1 of 5 ─────────────────────────────────────────────

2026-04-29 09:15:07,685 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 5] No timedelta provided, so the datetime was not advanced.
2026-04-29 09:15:11,058 - ThreadPoolExecutor-33_2(19308) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 09:15:11,076 - ThreadPoolExecutor-33_3(2744) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 09:15:11,143 - ThreadPoolExecutor-33_2(19308) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 09:15:11,160 - ThreadPoolExecutor-33_3(2744) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 09:15:11,595 - ThreadPoolExecutor-33_1(2412) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 09:15:11,603 - ThreadPoolExecutor-33_0(41160) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 09:15:11,697 - ThreadPoolExecutor-33_1(2412) - tinytroupe - INFO - Wait

...

──────────────────────────────────────────── TinyWorld 10 step 4 of 5 ─────────────────────────────────────────────

2026-04-29 10:37:30,294 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 10] No timedelta provided, so the datetime was not advanced.
2026-04-29 10:37:32,082 - ThreadPoolExecutor-76_1(25380) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 10:37:32,101 - ThreadPoolExecutor-76_2(1820) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 10:37:32,126 - ThreadPoolExecutor-76_0(35160) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 10:37:32,132 - ThreadPoolExecutor-76_3(15996) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 10:37:32,149 - ThreadPoolExecutor-76_1(25380) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 10:37:32,168 - ThreadPoolExecutor-76_2(1820) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 10:37:32,185 - ThreadPoolExecutor-76_0(35160) - tinytroupe - INFO - W

──────────────────────────────────────────── TinyWorld 10 step 5 of 5 ─────────────────────────────────────────────

2026-04-29 10:38:37,925 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 10] No timedelta provided, so the datetime was not advanced.
2026-04-29 10:38:39,534 - ThreadPoolExecutor-77_2(13624) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 10:38:39,553 - ThreadPoolExecutor-77_1(40704) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 10:38:39,567 - ThreadPoolExecutor-77_0(11448) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 10:38:39,571 - ThreadPoolExecutor-77_3(33408) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 10:38:39,594 - ThreadPoolExecutor-77_2(13624) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 10:38:39,618 - ThreadPoolExecutor-77_1(40704) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 10:38:39,634 - ThreadPoolExecutor-77_3(33408) - tinytroupe - INFO -

({'Hard Persona Adherence': [3,
   3,
   0,
   4,
   0,
   1,
   1,
   5,
   1,
   3,
   0,
   5,
   3,
   0,
   1,
   4,
   2,
   2,
   0,
   3,
   2,
   0,
   0,
   3,
   4,
   3,
   2,
   4,
   2,
   2,
   2,
   3,
   1,
   0,
   0,
   6,
   3,
   0,
   3,
   3],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9],
  'Fluency': [7,
   7,
   8,
   1,
   7,
   7,
   8,
   8,
   8,
   8,
   8,
   8,
   7,
   9,
   9,
   7,
   8,
   9,
   9,
   8,
   8,
   7,
   8,
   8,
   9,
   9,
   9,
   9,
   8,
   8,
   8,
   7,
   8,
   8,
   8,
   1,
   8,
   8,
   8,
   8]},
 {'ideas_qty': [4, 4, 3, 4, 4, 4, 4, 4, 4, 4],
  'Task Completion': [9, 9, 9, 9, 9, 9, 9, 9, 9, 9],
  'Divergence': [1, 0, 0, 0, 3, 1, 0, 0, 0, 0]})

In [18]:
brainstorm(people_groups[1], proposals_groups[0]) if len(people_groups) > 1  and len(proposals_groups) > 0 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 4
Discussion objective: Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.
Trial number: 1
Agents: [TinyPerson(name='Colin Arthur Matthews'), TinyPerson(name='Colin Murray'), TinyPerson(name='Connor Walsh'), TinyPerson(name='Darren McCall')]
2026-04-29 10:48:51,186 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 11] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 11 step 1 of 1 ─────────────────────────────────────────────

2026-04-29 10:48:51,193 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 11] No timedelta provided, so the datetime was not advanced.
2026-04-29 10:48:52,856 - ThreadPoolExecutor-80_2(33856) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 10:48:52,861 - ThreadPoolExecutor-80_3(32596) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 10:48:52,861 - ThreadPoolExecutor-80_0(40676) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 10:48:52,870 - ThreadPoolExecutor-80_1(19744) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 10:48:52,903 - ThreadPoolExecutor-80_2(33856) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 10:48:52,924 - ThreadPoolExecutor-80_3(32596) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 10:48:52,930 - ThreadPoolExecutor-80_0(40676) - tinytroupe - INFO -

──────────────────────────────────────────── TinyWorld 11 step 1 of 5 ─────────────────────────────────────────────

2026-04-29 10:49:54,745 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 11] No timedelta provided, so the datetime was not advanced.
2026-04-29 10:49:56,462 - ThreadPoolExecutor-81_3(5300) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 10:49:56,480 - ThreadPoolExecutor-81_1(35196) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 10:49:56,499 - ThreadPoolExecutor-81_0(31720) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 10:49:56,512 - ThreadPoolExecutor-81_2(37180) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 10:49:56,521 - ThreadPoolExecutor-81_3(5300) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 10:49:56,534 - ThreadPoolExecutor-81_1(35196) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 10:49:56,551 - ThreadPoolExecutor-81_0(31720) - tinytroupe - INFO - W

...

──────────────────────────────────────────── TinyWorld 14 step 4 of 5 ─────────────────────────────────────────────

2026-04-29 11:42:13,391 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 14] No timedelta provided, so the datetime was not advanced.
2026-04-29 11:42:14,929 - ThreadPoolExecutor-108_1(5472) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 11:42:14,940 - ThreadPoolExecutor-108_3(14764) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 11:42:14,944 - ThreadPoolExecutor-108_2(25292) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 11:42:14,955 - ThreadPoolExecutor-108_0(38440) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 11:42:14,977 - ThreadPoolExecutor-108_1(5472) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 11:42:14,996 - ThreadPoolExecutor-108_2(25292) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 11:42:15,001 - ThreadPoolExecutor-108_0(38440) - tinytroupe - I

──────────────────────────────────────────── TinyWorld 14 step 5 of 5 ─────────────────────────────────────────────

2026-04-29 11:43:14,381 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 14] No timedelta provided, so the datetime was not advanced.
2026-04-29 11:43:16,392 - ThreadPoolExecutor-109_0(13140) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 11:43:16,449 - ThreadPoolExecutor-109_0(13140) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 11:43:16,496 - ThreadPoolExecutor-109_1(28252) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 11:43:16,513 - ThreadPoolExecutor-109_3(37644) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 11:43:16,530 - ThreadPoolExecutor-109_2(21948) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 11:43:16,546 - ThreadPoolExecutor-109_1(28252) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 11:43:16,568 - ThreadPoolExecutor-109_3(37644) - tinytroupe -

({'Hard Persona Adherence': [3,
   3,
   0,
   4,
   0,
   1,
   1,
   5,
   1,
   3,
   0,
   5,
   3,
   0,
   1,
   4,
   2,
   2,
   0,
   3,
   2,
   0,
   0,
   3,
   4,
   3,
   2,
   4,
   2,
   2,
   2,
   3,
   1,
   0,
   0,
   6,
   3,
   0,
   3,
   3,
   1,
   6,
   1,
   1,
   0,
   0,
   1,
   2,
   0,
   0,
   0,
   0,
   5,
   0,
   3,
   1],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9],
  'Fluency': [7,
   7,
   8,
   1,
   7,
   7,
   8,
   8,
   8,
   8,
   8,
   8,
   7,
   9,
   9,
   7,
   8,
   9,
   9,
   8,
   8,
   7,
   8,
   8,
   9,
   9,
   9,
   9,
   8,
   8,
   8,
   7,
   8,
   8,
   8,
   1,
   8,
   8,
   8,
   8,
   9,
   7,
   8,
   9,
   8,

In [19]:
brainstorm(people_groups[1], proposals_groups[1]) if len(people_groups) > 1  and len(proposals_groups) > 1 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 6
Discussion objective: Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.
Trial number: 1
Agents: [TinyPerson(name='Colin Arthur Matthews'), TinyPerson(name='Colin Murray'), TinyPerson(name='Connor Walsh'), TinyPerson(name='Darren McCall')]
2026-04-29 11:53:47,255 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 15] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 15 step 1 of 1 ─────────────────────────────────────────────

2026-04-29 11:53:47,263 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 15] No timedelta provided, so the datetime was not advanced.
2026-04-29 11:53:49,751 - ThreadPoolExecutor-112_3(736) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 11:53:49,779 - ThreadPoolExecutor-112_2(2372) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 11:53:49,784 - ThreadPoolExecutor-112_1(43660) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 11:53:49,798 - ThreadPoolExecutor-112_0(42752) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 11:53:49,828 - ThreadPoolExecutor-112_3(736) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 11:53:49,850 - ThreadPoolExecutor-112_2(2372) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 11:53:49,857 - ThreadPoolExecutor-112_1(43660) - tinytroupe - INFO 

──────────────────────────────────────────── TinyWorld 15 step 1 of 5 ─────────────────────────────────────────────

2026-04-29 11:54:47,102 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 15] No timedelta provided, so the datetime was not advanced.
2026-04-29 11:54:48,758 - ThreadPoolExecutor-113_3(43796) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 11:54:48,769 - ThreadPoolExecutor-113_0(908) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 11:54:48,774 - ThreadPoolExecutor-113_1(23248) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 11:54:48,793 - ThreadPoolExecutor-113_2(41840) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 11:54:48,814 - ThreadPoolExecutor-113_0(908) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 11:54:48,819 - ThreadPoolExecutor-113_3(43796) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 11:54:48,829 - ThreadPoolExecutor-113_1(23248) - tinytroupe - INF

...

──────────────────────────────────────────── TinyWorld 20 step 4 of 5 ─────────────────────────────────────────────

2026-04-29 13:12:09,814 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 20] No timedelta provided, so the datetime was not advanced.
2026-04-29 13:12:12,835 - ThreadPoolExecutor-156_1(34564) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 13:12:12,898 - ThreadPoolExecutor-156_1(34564) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 13:12:12,933 - ThreadPoolExecutor-156_2(35952) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 13:12:12,974 - ThreadPoolExecutor-156_3(43548) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 13:12:12,982 - ThreadPoolExecutor-156_0(43596) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 13:12:13,022 - ThreadPoolExecutor-156_2(35952) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 13:12:13,053 - ThreadPoolExecutor-156_3(43548) - tinytroupe -

──────────────────────────────────────────── TinyWorld 20 step 5 of 5 ─────────────────────────────────────────────

2026-04-29 13:13:24,168 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 20] No timedelta provided, so the datetime was not advanced.
2026-04-29 13:13:26,106 - ThreadPoolExecutor-157_1(41088) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 13:13:26,112 - ThreadPoolExecutor-157_0(32728) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 13:13:26,154 - ThreadPoolExecutor-157_2(21752) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 13:13:26,172 - ThreadPoolExecutor-157_3(22352) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 13:13:26,196 - ThreadPoolExecutor-157_0(32728) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 13:13:26,201 - ThreadPoolExecutor-157_1(41088) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 13:13:26,244 - ThreadPoolExecutor-157_2(21752) - tinytroupe -

({'Hard Persona Adherence': [3,
   3,
   0,
   4,
   0,
   1,
   1,
   5,
   1,
   3,
   0,
   5,
   3,
   0,
   1,
   4,
   2,
   2,
   0,
   3,
   2,
   0,
   0,
   3,
   4,
   3,
   2,
   4,
   2,
   2,
   2,
   3,
   1,
   0,
   0,
   6,
   3,
   0,
   3,
   3,
   1,
   6,
   1,
   1,
   0,
   0,
   1,
   2,
   0,
   0,
   0,
   0,
   5,
   0,
   3,
   1,
   1,
   2,
   1,
   3,
   0,
   2,
   2,
   0,
   2,
   3,
   2,
   3,
   2,
   1,
   3,
   2,
   0,
   1,
   3,
   0,
   2,
   0,
   0,
   0],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,

In [20]:
brainstorm(people_groups[2], proposals_groups[0]) if len(people_groups) > 2  and len(proposals_groups) > 0 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 4
Discussion objective: Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.
Trial number: 1
Agents: [TinyPerson(name='Dean Bartlett'), TinyPerson(name='Declan Blackwell'), TinyPerson(name='Edgar Milton Crane'), TinyPerson(name='Leonard Victor Hale')]
2026-04-29 13:24:54,799 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 21] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 21 step 1 of 1 ─────────────────────────────────────────────

2026-04-29 13:24:54,812 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 21] No timedelta provided, so the datetime was not advanced.
2026-04-29 13:24:59,163 - ThreadPoolExecutor-160_1(37076) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 13:24:59,201 - ThreadPoolExecutor-160_2(32816) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 13:24:59,289 - ThreadPoolExecutor-160_1(37076) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 13:24:59,330 - ThreadPoolExecutor-160_2(32816) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 13:24:59,765 - ThreadPoolExecutor-160_3(8280) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 13:24:59,859 - ThreadPoolExecutor-160_3(8280) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 13:24:59,880 - ThreadPoolExecutor-160

──────────────────────────────────────────── TinyWorld 21 step 1 of 5 ─────────────────────────────────────────────

2026-04-29 13:25:51,828 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 21] No timedelta provided, so the datetime was not advanced.
2026-04-29 13:25:56,634 - ThreadPoolExecutor-161_1(34308) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 13:25:56,644 - ThreadPoolExecutor-161_0(25820) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 13:25:56,755 - ThreadPoolExecutor-161_1(34308) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 13:25:56,784 - ThreadPoolExecutor-161_0(25820) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 13:25:57,091 - ThreadPoolExecutor-161_2(43968) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 13:25:57,187 - ThreadPoolExecutor-161_3(7080) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 13:25:57,235 - ThreadPoolExecutor-161_2(43968) - tinytroupe - 

...

──────────────────────────────────────────── TinyWorld 24 step 4 of 5 ─────────────────────────────────────────────

2026-04-29 14:13:39,805 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 24] No timedelta provided, so the datetime was not advanced.
2026-04-29 14:13:44,353 - ThreadPoolExecutor-188_1(15292) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 14:13:44,460 - ThreadPoolExecutor-188_1(15292) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 14:13:44,468 - ThreadPoolExecutor-188_0(41696) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 14:13:44,642 - ThreadPoolExecutor-188_0(41696) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 14:13:44,739 - ThreadPoolExecutor-188_3(43784) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 14:13:44,764 - ThreadPoolExecutor-188_2(29224) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 14:13:45,012 - ThreadPoolExecutor-188_3(43784) - tinytroupe -

──────────────────────────────────────────── TinyWorld 24 step 5 of 5 ─────────────────────────────────────────────

2026-04-29 14:14:31,933 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 24] No timedelta provided, so the datetime was not advanced.
2026-04-29 14:14:36,292 - ThreadPoolExecutor-189_0(29652) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 14:14:36,357 - ThreadPoolExecutor-189_1(42316) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 14:14:36,451 - ThreadPoolExecutor-189_0(29652) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 14:14:36,490 - ThreadPoolExecutor-189_1(42316) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 14:14:36,640 - ThreadPoolExecutor-189_2(27084) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 14:14:36,733 - ThreadPoolExecutor-189_3(1572) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 14:14:36,845 - ThreadPoolExecutor-189_2(27084) - tinytroupe - 

({'Hard Persona Adherence': [3,
   3,
   0,
   4,
   0,
   1,
   1,
   5,
   1,
   3,
   0,
   5,
   3,
   0,
   1,
   4,
   2,
   2,
   0,
   3,
   2,
   0,
   0,
   3,
   4,
   3,
   2,
   4,
   2,
   2,
   2,
   3,
   1,
   0,
   0,
   6,
   3,
   0,
   3,
   3,
   1,
   6,
   1,
   1,
   0,
   0,
   1,
   2,
   0,
   0,
   0,
   0,
   5,
   0,
   3,
   1,
   1,
   2,
   1,
   3,
   0,
   2,
   2,
   0,
   2,
   3,
   2,
   3,
   2,
   1,
   3,
   2,
   0,
   1,
   3,
   0,
   2,
   0,
   0,
   0,
   2,
   0,
   0,
   1,
   2,
   3,
   0,
   0,
   0,
   1,
   3,
   0,
   0,
   3,
   3,
   0],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,

In [21]:
brainstorm(people_groups[2], proposals_groups[1]) if len(people_groups) > 2  and len(proposals_groups) > 1 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 6
Discussion objective: Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.
Trial number: 1
Agents: [TinyPerson(name='Dean Bartlett'), TinyPerson(name='Declan Blackwell'), TinyPerson(name='Edgar Milton Crane'), TinyPerson(name='Leonard Victor Hale')]
2026-04-29 14:24:58,470 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 25] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 25 step 1 of 1 ─────────────────────────────────────────────

2026-04-29 14:24:58,479 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 25] No timedelta provided, so the datetime was not advanced.
2026-04-29 14:25:01,847 - ThreadPoolExecutor-192_1(24056) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 14:25:01,984 - ThreadPoolExecutor-192_1(24056) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 14:25:01,991 - ThreadPoolExecutor-192_2(23872) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 14:25:02,087 - ThreadPoolExecutor-192_2(23872) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 14:25:02,235 - ThreadPoolExecutor-192_3(43560) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 14:25:02,251 - ThreadPoolExecutor-192_0(36964) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 14:25:02,383 - ThreadPoolExecutor-192_3(43560) - tinytroupe -

──────────────────────────────────────────── TinyWorld 25 step 1 of 5 ─────────────────────────────────────────────

2026-04-29 14:25:57,200 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 25] No timedelta provided, so the datetime was not advanced.
2026-04-29 14:26:02,148 - ThreadPoolExecutor-193_1(29228) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 14:26:02,179 - ThreadPoolExecutor-193_0(38908) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 14:26:02,235 - ThreadPoolExecutor-193_1(29228) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 14:26:02,279 - ThreadPoolExecutor-193_0(38908) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 14:26:02,414 - ThreadPoolExecutor-193_3(34784) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 14:26:02,477 - ThreadPoolExecutor-193_2(27968) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 14:26:02,518 - ThreadPoolExecutor-193_3(34784) - tinytroupe -

...

──────────────────────────────────────────── TinyWorld 30 step 4 of 5 ─────────────────────────────────────────────

2026-04-29 15:38:46,105 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 30] No timedelta provided, so the datetime was not advanced.
2026-04-29 15:38:51,851 - ThreadPoolExecutor-236_0(14644) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 15:38:52,000 - ThreadPoolExecutor-236_1(20220) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 15:38:52,036 - ThreadPoolExecutor-236_0(14644) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 15:38:52,131 - ThreadPoolExecutor-236_1(20220) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 15:38:54,926 - ThreadPoolExecutor-236_2(15264) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 15:38:55,263 - ThreadPoolExecutor-236_3(25296) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 15:38:55,376 - ThreadPoolExecutor-236_2(15264) - tinytroupe -

──────────────────────────────────────────── TinyWorld 30 step 5 of 5 ─────────────────────────────────────────────

2026-04-29 15:39:39,111 - MainThread(23188) - tinytroupe - INFO - [TinyWorld 30] No timedelta provided, so the datetime was not advanced.
2026-04-29 15:39:45,106 - ThreadPoolExecutor-237_1(34272) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 15:39:45,130 - ThreadPoolExecutor-237_0(38244) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 15:39:45,359 - ThreadPoolExecutor-237_1(34272) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 15:39:45,415 - ThreadPoolExecutor-237_0(38244) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-29 15:39:48,251 - ThreadPoolExecutor-237_3(17376) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 15:39:48,346 - ThreadPoolExecutor-237_2(22372) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-29 15:39:48,545 - ThreadPoolExecutor-237_3(17376) - tinytroupe -

({'Hard Persona Adherence': [3,
   3,
   0,
   4,
   0,
   1,
   1,
   5,
   1,
   3,
   0,
   5,
   3,
   0,
   1,
   4,
   2,
   2,
   0,
   3,
   2,
   0,
   0,
   3,
   4,
   3,
   2,
   4,
   2,
   2,
   2,
   3,
   1,
   0,
   0,
   6,
   3,
   0,
   3,
   3,
   1,
   6,
   1,
   1,
   0,
   0,
   1,
   2,
   0,
   0,
   0,
   0,
   5,
   0,
   3,
   1,
   1,
   2,
   1,
   3,
   0,
   2,
   2,
   0,
   2,
   3,
   2,
   3,
   2,
   1,
   3,
   2,
   0,
   1,
   3,
   0,
   2,
   0,
   0,
   0,
   2,
   0,
   0,
   1,
   2,
   3,
   0,
   0,
   0,
   1,
   3,
   0,
   0,
   3,
   3,
   0,
   2,
   3,
   1,
   1,
   1,
   3,
   2,
   1,
   3,
   5,
   2,
   0,
   1,
   4,
   3,
   1,
   2,
   0,
   3,
   1,
   0,
   1,
   2,
   1],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,

In [22]:
brainstorm(people_groups[3], proposals_groups[0]) if len(people_groups) > 3  and len(proposals_groups) > 0 else None

In [23]:
brainstorm(people_groups[3], proposals_groups[1]) if len(people_groups) > 3  and len(proposals_groups) > 1 else None

In [24]:
brainstorm(people_groups[4], proposals_groups[0]) if len(people_groups) > 4  and len(proposals_groups) > 0 else None

In [25]:
brainstorm(people_groups[4], proposals_groups[1]) if len(people_groups) > 4  and len(proposals_groups) > 1 else None

## Extract results and analyze

In [26]:
if experiment_runner.get_active_experiment() in ["Control", "Treatment"]:
    combined_scores = {**agent_propositions_scores, **environment_propositions_scores}
    experiment_runner.add_experiment_results(combined_scores, experiment_name=experiment_runner.get_active_experiment()) 
    
    plot_scores(combined_scores)

else:
    print("Experiment finished. No more experiments to run.")

{'Divergence': [1,
                0,
                0,
                0,
                3,
                1,
                0,
                0,
                0,
                0,
                0,
                2,
                3,
                8,
                4,
                0,
                0,
                0,
                5,
                2,
                0,
                0,
                0,
                0,
                1,
                2,
                0,
                0,
                0,
                2],
 'Fluency': [7,
             7,
             8,
             1,
             7,
             7,
             8,
             8,
             8,
             8,
             8,
             8,
             7,
             9,
             9,
             7,
             8,
             9,
             9,
             8,
             8,
             7,
             8,
             8,
             9,
             9,
             

,Proposition,Average Score,Standard Deviation,Count
0,Hard Persona Adherence,1.675000,1.496003,120.0
1,Self-consistency,8.733333,1.034963,120.0
2,Fluency,7.650000,1.673571,120.0
3,ideas_qty,4.071429,0.766356,28.0
4,Task Completion,8.933333,0.365148,30.0
5,Divergence,1.133333,1.888866,30.0


In [27]:
if experiment_runner.has_finished_all_experiments():
    print("All experiments have been finished.")
    print(f"STATISTICTS: Control vs")
    pprint(experiment_runner.run_statistical_tests(control_experiment_name='Control'))

    # plot scores of both experiments
    experiment_control_scores = experiment_runner.get_experiment_results("Control")
    experiment_treatment_scores = experiment_runner.get_experiment_results("Treatment")
    
    
    plot_scores(experiment_control_scores)
    plot_scores(experiment_treatment_scores)

else:
    print("Not all experiments have been finished. RESTART AND RERUN.")

Not all experiments have been finished. RESTART AND RERUN.


In [28]:
experiment_runner.finish_active_experiment()

2026-04-29 15:49:16,846 - MainThread(23188) - tinytroupe - INFO - Experiment 'Control' marked as finished.


True